In [1]:
import pandas
import matplotlib.pyplot as plt
import numpy
import math
import os
from typing import List

In [2]:
def oneD_GaussLegendreQuadrature(n):
    N = n - 1
    N1 = n
    N2 = n + 1
    xu = numpy.linspace(-1, 1, N1).reshape(-1,1)
    x= - numpy.cos((2*(numpy.linspace(0,N,N1).reshape(-1,1)) + 1)*numpy.pi/(2*N+2)) + (0.27/N1)*numpy.sin(numpy.pi*xu*N/N2)
    L = numpy.zeros((n, n+1))
    x0 = 2

    while numpy.max(numpy.absolute(x-x0)) > numpy.finfo(numpy.float64).eps:
        L[:, 0] = 1
        L[: , 1] = numpy.transpose(x)
        for i in range(1,n):
            temp_Li = ((2*i+1)*(x.T*L[:,i]) - i*L[:,i-1])/(i+1)
            L[:, i+1] = numpy.reshape(temp_Li, (1,-1))
        temp_dLp = ((n)*(x.T*L[:, n]) - (n)*L[:, n-1])/((x.T*x.T) - 1)
        dLp = temp_dLp
        x0 = x
        x = x0 - numpy.reshape((L[:, n]/dLp),(-1,1))
    w = (2/((1-(x.T*x.T))*(dLp*dLp)))
    return x.T, w

class legendreGaussQuad():
    def __init__(self, ngp, dim):
        self.gps1d, self.w1d = oneD_GaussLegendreQuadrature(ngp)
        gps = numpy.zeros((ngp**dim, dim))
        gps[:, 0] = numpy.tile(self.gps1d, ngp**(dim-1))
        if dim == 3:
            gps[:, 1] = numpy.tile(numpy.repeat(self.gps1d,ngp),ngp)
            gps[:, 2] = numpy.repeat(self.gps1d, ngp**(dim-1))
        else:
            gps[:, 1] = numpy.repeat(self.gps1d, ngp**(dim-1))
        self.gps = gps
        self.ngp = ngp**dim
        w1 = numpy.tile(self.w1d, ngp**(dim-1))
        w2 = numpy.repeat(self.w1d, ngp**(dim-1))
        self.w = (w1*w2).T

In [3]:
#   x ---------------------- x
#  -1                        1
def shapeL1(xi):
    N = [0.5 - xi/2, xi/2 + 0.5]
    dpN = [-0.5, 0.5]
    return N, dpN
#   x ----------- x ----------- x
#  -1             0             1
def shapeL2(xi):
    N = [(xi*(xi - 1))/2, -(xi - 1)*(xi + 1), xi*(xi/2 + 1/2)]
    dpN = [ xi - 1/2, -2*xi, xi + 1/2 ]
    return N, dpN
#   x ----- x ----- x ----- x
#  -1     -.33     .33      1
def shapeL3(xi):
    N = [ -(3*((3*xi)/2 + 1/2)*(xi - 1)*(xi - 1/3))/8,
    (9*((3*xi)/2 + 3/2)*(xi - 1)*(xi - 1/3))/8,
    -(9*((3*xi)/4 + 3/4)*(xi - 1)*(xi + 1/3))/4,
    (9*(xi/2 + 1/2)*(xi - 1/3)*(xi + 1/3))/8 ]

    dpN = [ (9*xi)/8 - (27*xi^2)/16 + 1/16,
    (81*xi^2)/16 - (9*xi)/8 - 27/16,
    27/16 - (81*xi^2)/16 - (9*xi)/8,
    (27*xi^2)/16 + (9*xi)/8 - 1/16 ]

    return N, dpN

def shapesLinear(xi, p):
    N = numpy.zeros(((p+1), 1))
    dpN = numpy.zeros(((p+1), 1))
    if p == 1:
        N, dpN = shapeL1(xi)
    elif p == 2:
        N, dpN = shapeL2(xi)
    elif p == 3:
        N, dpN = shapeL3(xi)
    return N, dpN

def shapesQuad(xi, eta, p, q):
    N1, dpN1 = shapesLinear(xi, p)
    N2, dpN2 = shapesLinear(eta, q)

    dpN = numpy.zeros((2, (p+1)*(q+1)))
    N = numpy.reshape(numpy.outer(N1, N2), ((p + 1) * (q + 1), 1))
    dpN[0, :] = numpy.reshape(numpy.outer(N2, dpN1), ((p + 1) * (q + 1)))
    dpN[1, :] = numpy.reshape(numpy.outer(dpN2, N1), ((p + 1) * (q + 1)))

    return N, dpN

class mesh_element():
    def __init__(self, x: List[List[float]] = None):
        self.con_XIs = x
        self.xIs = {}
        self.Boundary = []
        self.p = None #order of approximation
        
class Node():
    def __init__(self, x, y):
        self.X = numpy.zeros((len(x), 2))
        self.Boundary = numpy.zeros((len(x), 1))
        
        for idx, xy in enumerate(list(zip(x,y))):
            self.X[idx, 0] = xy[0]
            self.X[idx, 1] = xy[1]
            if xy[0] == max(x) or xy[0] == min(x) or xy[1] == min(y) or xy[1] == max(y):
                self.Boundary[idx] = 1
            else:
                self.Boundary[idx] = 0

In [4]:
quadr = legendreGaussQuad(ngp= 15, dim= 2)

def analytical_solution(x,y, boundary):
    a= 1
    b= 1
    BC= 300
    if boundary == 1:
        if y == 0:
            soln = 300
        if y == 1:
            soln = 0
        if x == 0:
            soln = 0
        if x == 1:
            soln = 0
    elif boundary == 0:
        SUM = 0
        errList = []
        for n in range(0,100):
            try:
                A = math.sin((2*n+1)*(math.pi*x)/a)
                B = math.sinh(((b-y)*(2*n+1)*math.pi/a))
                C = math.sinh((2*n+1)*math.pi*b/a)
                SUM += (1/((2*n)+1)) * (A * B / C) 
            except (RuntimeWarning, OverflowError) as e:
                errList.append(n)
                continue
        soln= (4*BC/math.pi)*SUM
    return soln

In [5]:
XIs = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
YIs = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
elem = []
num_el = 0
for idy, y in enumerate(YIs):
    if y != YIs[-1]:
        for idx, x in enumerate(XIs):
            if x != XIs[-1]:
                XI_Con = [[XIs[idx], YIs[idy]], [XIs[idx+1], YIs[idy]], [XIs[idx], YIs[idy+1]], [XIs[idx+1], YIs[idy+1]]]
                cur_el = mesh_element(XI_Con)
                elem.append(cur_el)
                num_el += 1
            

In [6]:
df = pandas.read_csv(r'C:\Users\harsh\Documents\thesis\output_poisson.csv')
df_x = df['X (m)'].values.round(decimals= 2).tolist()
df_y = df['Y (m)'].values.round(decimals= 2).tolist()
df_T = df['Temperature (K)'].values.round(decimals= 4).tolist()
df_xy = numpy.zeros((len(df_x), 3))
df_xy[:, 0] = df_x
df_xy[:, 1] = df_y
df_xy[:, 2] = df_T
#print(df_xy)


node = Node(df_x, df_y)

In [7]:
check = 0
for id, element in enumerate(elem):
    #print(elem[id].xIs)
    #print(element.con_XIs)
    xIs = []
    boundary_data = []
    for con_XI in element.con_XIs:
        xy = [con_XI[0], con_XI[1]]
        #print(xy)
        for idx, xyT in enumerate(df_xy):
            if xy == [df_xy[idx, 0], df_xy[idx,1]]:
                #print(xyT)
                xIs.append(xyT[2])
        for idx, XY in enumerate(node.X):
            if xy == [XY[0], XY[1]]:
                if node.Boundary[idx] == 1:
                    boundary_data.append(1)
                else:
                    boundary_data.append(0)       
    elem[id].xIs = xIs
    elem[id].Boundary = boundary_data


In [8]:
L_inf = []
for idx, xy in enumerate(node.X):
    f_analytical = analytical_solution(x= xy[0], y= xy[1], boundary= node.Boundary[idx])
    for id, xyT in enumerate(df_xy):
        if [xy[0], xy[1]] == [xyT[0], xyT[1]]:
            f_numerical = xyT[2]
    L_inf.append(abs(float(f_analytical) - float(f_numerical)))
        

In [9]:
print(max(L_inf))

150.0


In [10]:
L2_error = 0
pp = 1
num_el = 0
for element in elem:
    f_analytical = []
    for idx, con_XI in enumerate(element.con_XIs):
        an_soln = analytical_solution(x= con_XI[0], y= con_XI[1], boundary= element.Boundary[idx])
        f_analytical.append(an_soln)
        #bprint(con_XI, an_soln, element.xIs[idx])
    L2_el = 0

    for gp in range(quadr.ngp):
        xi = quadr.gps[gp, 0]
        eta = quadr.gps[gp, 1]
        N, dpN = shapesQuad(xi= xi, eta= eta, p= pp, q= pp)
        J = numpy.matmul(dpN, element.con_XIs)
        dN_dx = numpy.linalg.inv(J)@dpN
        err = abs(N.T@f_analytical - N.T@element.xIs)
        L2_el = L2_el + (err**2)*(numpy.linalg.det(J))*quadr.w[gp]
    #print(L2_el) 
    L2_error = L2_error + L2_el
    num_el += 1

In [11]:
print(numpy.sqrt(L2_error/num_el))

[0.70900285]


In [12]:
import pyvista as pv
filename = r'C:\Users\harsh\Documents\thesis\poisson_finer.cgns'
reader = pv.CGNSReader(filename)

In [13]:
reader.cell_array_names

[]

In [14]:
ds = reader.read()